In [ ]:
# 运行准备：使用p5lib按示例代码5.1～5.3组织数据和fe，并使用示例代码5.43、5.45定义FeatSet和SimpleNet
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from p5lib.ch5 import FeatSet, SimpleNet, build_bow_features, load_imdb_data
import numpy as np

sents_train, sents_test, y_train, y_test = load_imdb_data()
fe, vob, x_train, x_test = build_bow_features(sents_train, sents_test)
train_ds = FeatSet(fe.transform(sents_train), y_train)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_ds = FeatSet(fe.transform(sents_test), [0]*len(sents_test))
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
vocab_size = len(vob)

model = SimpleNet(vocab_size)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = nn.CrossEntropyLoss()
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)
max_epoch = 300
total_steps = len(train_loader) * max_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

## 训练SimpleNet

In [2]:
import time

t0 = time.perf_counter() # 记录开始时刻
for epoch in range(max_epoch):
    model.train()
    total_loss = 0
    for batch in train_loader:
        xb, yb = batch
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    total_loss /= len(train_loader) # 注意要在内循环之外
    if epoch % 20 == 0: # 每20轮次打印一次训练损失
        print(f"epoch {epoch:02d} : Loss = {total_loss:.4f}")
if device.type == "cuda":
    torch.cuda.synchronize(device)
total_time = time.perf_counter() - t0 # 训练结束计时
print(f"Training finished in {total_time/60:.2f} min")

epoch 00 : Loss = 0.6933


epoch 20 : Loss = 0.6435


epoch 40 : Loss = 0.6033


epoch 60 : Loss = 0.5713


epoch 80 : Loss = 0.5445


...

epoch 220 : Loss = 0.4453


epoch 240 : Loss = 0.4397


epoch 260 : Loss = 0.4362


epoch 280 : Loss = 0.4325


Training finished in 0.16 min


## SimpleNet性能评测

In [3]:
from sklearn.metrics import classification_report

model.eval() #将模型设置为推理模式
all_preds = []
with torch.no_grad(): # 禁用梯度计算，提高推理效率
    for xb, _ in test_loader:
        xb = xb.to(device)
        output = model(xb)
        preds = torch.argmax(output, dim=1).cpu().numpy() # 每个样本取得分最高的类别作为预测结果
        all_preds.extend(preds)
# 性能评测
print(classification_report(y_test, all_preds, digits=3))

              precision    recall  f1-score   support

           0      0.840     0.848     0.844       105
           1      0.830     0.821     0.825        95

    accuracy                          0.835       200
   macro avg      0.835     0.834     0.834       200
weighted avg      0.835     0.835     0.835       200



## 用额外样本测试SimpleNet

In [4]:
input_texts = ['this is a good movie', 'the movie is horrible', 'a horrible movie', 'not a bad movie']
input_ds = FeatSet(fe.transform(input_texts), [0]*len(input_texts))
input_loader = DataLoader(input_ds, batch_size=16, shuffle=False)
model.eval()
all_scores = []
with torch.no_grad():
    for batch in input_loader:
        xb, yb = batch
        xb = xb.to(device)
        yb = yb.to(device)
        output = model(xb)
        scores = torch.softmax(output, dim=1).cpu().numpy()
        all_scores.append(scores)
scores_simplenet = np.vstack(all_scores)
print(scores_simplenet)

[[0.48685423 0.51314574]
 [0.5617901  0.43820995]
 [0.5661715  0.4338285 ]
 [0.64710534 0.35289457]]
